# T3: Exploratory data analysis - Visualizations
## NYC TLC Trip Record Data

This notebook visualizes the EDA results from all four datasets:
- **Yellow taxi** (traditional medallion taxis)
- **Green taxi** (boro taxis, outer boroughs)
- **FHV** (for-hire vehicles, e.g. black cars, limos)
- **FHVHV** (high-volume for-hire vehicles: Uber, Lyft)

In [ ]:
%%sql -r dataset_summary
USE WAREHOUSE BIGDATA_MZMB_WH

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATASET_COLORS = {
    'yellow': '#F5C518',
    'green': '#2ECC71',
    'fhv': '#9B59B6',
    'fhvhv': '#3498DB'
}
DATASET_LABELS = {
    'yellow': 'Yellow taxi',
    'green': 'Green taxi',
    'fhv': 'FHV',
    'fhvhv': 'FHVHV (Uber/Lyft)'
}

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 9
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 9

def load_table(name):
    return session.sql(f'SELECT * FROM BIGDATA_TAXI_MZMB.GOLD.{name}').to_pandas()

In [ ]:
dataset_summary = load_table('T3_DATASET_SUMMARY')
df = dataset_summary
print("\n=== Dataset overview ===")
for _, row in df.iterrows():
    print(f"  {DATASET_LABELS[row['DATASET']]}: {row['TOTAL_TRIPS']:,.0f} trips ({row['MIN_YEAR']}-{row['MAX_YEAR']})")
print(f"\n  Total across all datasets: {df['TOTAL_TRIPS'].sum():,.0f} trips")

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    [DATASET_LABELS[d] for d in df['DATASET']],
    df['TOTAL_TRIPS'],
    color=[DATASET_COLORS[d] for d in df['DATASET']],
    edgecolor='black', linewidth=0.5
)
for bar, count in zip(bars, df['TOTAL_TRIPS']):
    ax.text(bar.get_width() + 1e7, bar.get_y() + bar.get_height()/2,
            f'{count/1e9:.2f}B', va='center', fontsize=9)
ax.set_xlabel('Total trips')
ax.set_title('Total trip count by dataset')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
plt.tight_layout()
plt.show()

---
## 1. Temporal analysis

In [ ]:
monthly_trend = load_table('T3_MONTHLY_TREND')
monthly_trend['MONTH_DATE'] = pd.to_datetime(monthly_trend['MONTH_DATE'])

fig, ax = plt.subplots(figsize=(14, 5))
for dataset in ['yellow', 'green', 'fhv', 'fhvhv']:
    subset = monthly_trend[monthly_trend['DATASET'] == dataset].sort_values('MONTH_DATE')
    ax.plot(subset['MONTH_DATE'], subset['TRIP_COUNT'] / 1e6,
            color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset],
            linewidth=1.2, alpha=0.85)

ax.axvline(pd.Timestamp('2019-02-01'), color='orange', linestyle='--', alpha=0.6, label='FHVHV regulation (Feb 2019)')
ax.axvline(pd.Timestamp('2020-03-01'), color='red', linestyle='--', alpha=0.5, label='COVID-19 (Mar 2020)')
ax.set_xlabel('Date')
ax.set_ylabel('Trips (millions)')
ax.set_title('Monthly trip volume over time - all datasets')
ax.legend(loc='upper left', framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
hourly_norm = load_table('T3_HOURLY_PATTERN_NORMALIZED')

fig, ax = plt.subplots(figsize=(10, 5))
for dataset in ['yellow', 'green', 'fhv', 'fhvhv']:
    subset = hourly_norm[hourly_norm['DATASET'] == dataset].sort_values('PICKUP_HOUR')
    ax.plot(subset['PICKUP_HOUR'], subset['PCT_OF_TOTAL'],
            color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset],
            linewidth=2, marker='o', markersize=3)

ax.set_xlabel('Hour of day')
ax.set_ylabel('% of total trips')
ax.set_title('Hourly pickup distribution (normalized) - comparison across all datasets')
ax.set_xticks(range(0, 24))
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.5, 23.5)
plt.tight_layout()
plt.show()

In [ ]:
dow_data = load_table('T3_TRIPS_BY_DOW')
DOW_NAMES = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']

fig, ax = plt.subplots(figsize=(10, 5))
bar_width = 0.2
datasets = ['yellow', 'green', 'fhv', 'fhvhv']

for i, dataset in enumerate(datasets):
    subset = dow_data[dow_data['DATASET'] == dataset].sort_values('PICKUP_DOW')
    total = subset['TRIP_COUNT'].sum()
    pct = subset['TRIP_COUNT'] / total * 100
    x = np.arange(7) + i * bar_width
    ax.bar(x, pct, bar_width, color=DATASET_COLORS[dataset],
           label=DATASET_LABELS[dataset], edgecolor='black', linewidth=0.3)

ax.set_xlabel('Day of week')
ax.set_ylabel('% of weekly trips')
ax.set_title('Day-of-week distribution (normalized) - all datasets')
ax.set_xticks(np.arange(7) + bar_width * 1.5)
ax.set_xticklabels(DOW_NAMES)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

---
## 2. COVID-19 impact analysis

In [ ]:
covid_impact = load_table('T3_COVID_IMPACT')
MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
datasets = ['yellow', 'green', 'fhv', 'fhvhv']

for ax, dataset in zip(axes.flatten(), datasets):
    subset = covid_impact[covid_impact['DATASET'] == dataset].sort_values('PICKUP_MONTH')
    subset = subset.dropna(subset=['TRIPS_2019', 'TRIPS_2020'])
    x = np.arange(len(subset))
    width = 0.35
    ax.bar(x - width/2, subset['TRIPS_2019'] / 1e6, width, color=DATASET_COLORS[dataset], alpha=0.7, label='2019')
    ax.bar(x + width/2, subset['TRIPS_2020'] / 1e6, width, color=DATASET_COLORS[dataset], alpha=0.35, label='2020', hatch='//')
    ax.set_title(f'{DATASET_LABELS[dataset]} - 2019 vs 2020')
    ax.set_xticks(x)
    ax.set_xticklabels([MONTH_NAMES[int(m)-1] for m in subset['PICKUP_MONTH']])
    ax.set_ylabel('Trips (millions)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('COVID-19 impact: 2019 vs 2020 monthly trips', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for dataset in datasets:
    subset = covid_impact[covid_impact['DATASET'] == dataset].sort_values('PICKUP_MONTH')
    subset = subset.dropna(subset=['PCT_CHANGE'])
    ax.plot(subset['PICKUP_MONTH'], subset['PCT_CHANGE'],
            color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset],
            linewidth=2, marker='o', markersize=4)

ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Month')
ax.set_ylabel('% change (2020 vs 2019)')
ax.set_title('Percentage change in trips: 2020 vs 2019 by month')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTH_NAMES)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Spatial analysis

In [ ]:
top_pickup = load_table('T3_TOP_PICKUP_LOCATIONS')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
datasets = ['yellow', 'green', 'fhv', 'fhvhv']

for ax, dataset in zip(axes.flatten(), datasets):
    subset = top_pickup[top_pickup['DATASET'] == dataset].sort_values('TRIP_COUNT', ascending=False).head(15)
    y_pos = np.arange(len(subset))
    ax.barh(y_pos, subset['TRIP_COUNT'] / 1e6,
            color=DATASET_COLORS[dataset], edgecolor='black', linewidth=0.3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"Zone {int(loc)}" for loc in subset['PU_LOCATION_ID']], fontsize=8)
    ax.set_xlabel('Trips (millions)')
    ax.set_title(f'{DATASET_LABELS[dataset]} - top 15 pickup locations')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

In [ ]:
od_pairs = load_table('T3_TOP_OD_PAIRS')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
datasets = ['yellow', 'green', 'fhv', 'fhvhv']

for ax, dataset in zip(axes.flatten(), datasets):
    subset = od_pairs[od_pairs['DATASET'] == dataset].sort_values('TRIP_COUNT', ascending=False).head(10)
    labels = [f"{int(row['PU_LOCATION_ID'])} \u2192 {int(row['DO_LOCATION_ID'])}" for _, row in subset.iterrows()]
    y_pos = np.arange(len(subset))
    ax.barh(y_pos, subset['TRIP_COUNT'] / 1e6,
            color=DATASET_COLORS[dataset], edgecolor='black', linewidth=0.3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel('Trips (millions)')
    ax.set_title(f'{DATASET_LABELS[dataset]} - top 10 OD pairs')
    ax.invert_yaxis()
    ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

---
## 4. Trip characteristics

In [ ]:
trip_stats = load_table('T3_YELLOW_TRIP_STATS')
trip_stats['DATASET'] = 'yellow'
green_stats = load_table('T3_GREEN_TRIP_STATS')
green_stats['DATASET'] = 'green'
fhvhv_stats = load_table('T3_FHVHV_TRIP_STATS')
fhvhv_stats['DATASET'] = 'fhvhv'
trip_stats = pd.concat([trip_stats, green_stats, fhvhv_stats], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
metrics = [
    ('AVG_DISTANCE', 'Average trip distance (miles)'),
    ('AVG_DURATION_MIN', 'Average trip duration (minutes)'),
    ('AVG_FARE', 'Average fare ($)'),
    ('AVG_TOTAL', 'Average total amount ($)')
]

for ax, (col, title) in zip(axes.flatten(), metrics):
    for dataset in ['yellow', 'green', 'fhvhv']:
        subset = trip_stats[trip_stats['DATASET'] == dataset].copy()
        if col in subset.columns:
            subset = subset[subset[col].notna() & (subset[col] < 100)]
            subset = subset.sort_values('PICKUP_YEAR')
            ax.plot(subset['PICKUP_YEAR'], subset[col],
                    color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset],
                    linewidth=2, marker='o', markersize=3)
    ax.set_xlabel('Year')
    ax.set_title(title)
    ax.legend(fontsize=7, framealpha=0.9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Trip characteristics over time - yellow, green, FHVHV comparison', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Uber vs Lyft (FHVHV breakdown)

In [ ]:
uber_lyft = load_table('T3_FHVHV_UBER_VS_LYFT')
uber_lyft = uber_lyft[uber_lyft['COMPANY'].isin(['Uber', 'Lyft'])]
uber = uber_lyft[uber_lyft['COMPANY'] == 'Uber'].sort_values('PICKUP_YEAR')
lyft = uber_lyft[uber_lyft['COMPANY'] == 'Lyft'].sort_values('PICKUP_YEAR')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
metrics = [
    ('TRIP_COUNT', 'Trip count', 1e6, 'Millions'),
    ('AVG_MILES', 'Avg distance (miles)', 1, 'Miles'),
    ('AVG_DURATION_MIN', 'Avg duration (min)', 1, 'Minutes'),
    ('AVG_FARE', 'Avg fare ($)', 1, '$'),
    ('AVG_TIP', 'Avg tip ($)', 1, '$'),
    ('AVG_DRIVER_PAY', 'Avg driver pay ($)', 1, '$')
]

for ax, (col, title, div, unit) in zip(axes.flatten(), metrics):
    ax.plot(uber['PICKUP_YEAR'], uber[col] / div, color='#000000', label='Uber', linewidth=2, marker='s', markersize=4)
    ax.plot(lyft['PICKUP_YEAR'], lyft[col] / div, color='#FF00BF', label='Lyft', linewidth=2, marker='o', markersize=4)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Uber vs Lyft comparison over time', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
years = uber['PICKUP_YEAR'].values
uber_share = uber['TRIP_COUNT'].values / (uber['TRIP_COUNT'].values + lyft['TRIP_COUNT'].values) * 100
lyft_share = 100 - uber_share

ax.bar(years, uber_share, color='#000000', label='Uber', edgecolor='white', linewidth=0.5)
ax.bar(years, lyft_share, bottom=uber_share, color='#FF00BF', label='Lyft', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Market share (%)')
ax.set_title('Uber vs Lyft market share (FHVHV trips)')
ax.legend()
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

---
## 6. Shared rides analysis (FHVHV)

In [ ]:
shared_rides = load_table('T3_FHVHV_SHARED_RIDES')
shared_rides = shared_rides.sort_values('PICKUP_YEAR')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(shared_rides['PICKUP_YEAR'], shared_rides['PCT_SHARED_REQUESTED'],
         color='#3498DB', linewidth=2, marker='o', label='Shared requested')
ax1.plot(shared_rides['PICKUP_YEAR'], shared_rides['PCT_SHARED_MATCHED'],
         color='#2ECC71', linewidth=2, marker='s', label='Shared matched')
ax1.set_xlabel('Year')
ax1.set_ylabel('% of total trips')
ax1.set_title('Shared ride request and match rates over time')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(shared_rides['PICKUP_YEAR'], shared_rides['MATCH_RATE'],
        color='#E74C3C', edgecolor='black', linewidth=0.3)
ax2.set_xlabel('Year')
ax2.set_ylabel('Match rate (%)')
ax2.set_title('Shared ride match rate (matched / requested)')
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

---
## 7. Payment type distribution

In [ ]:
payment_dist = load_table('T3_PAYMENT_TYPE_DIST')
PAYMENT_LABELS = {0: 'Unknown', 1: 'Credit card', 2: 'Cash', 3: 'No charge', 4: 'Dispute', 5: 'Voided'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, dataset, title in [(ax1, 'yellow', 'Yellow taxi'), (ax2, 'green', 'Green taxi')]:
    subset = payment_dist[payment_dist['DATASET'] == dataset].copy()
    subset['LABEL'] = subset['PAYMENT_TYPE'].map(PAYMENT_LABELS)
    subset = subset.sort_values('TRIP_COUNT', ascending=True)
    total = subset['TRIP_COUNT'].sum()
    subset['PCT'] = subset['TRIP_COUNT'] / total * 100
    colors_map = {'Credit card': '#3498DB', 'Cash': '#2ECC71', 'No charge': '#F39C12',
                  'Dispute': '#E74C3C', 'Voided': '#95A5A6', 'Unknown': '#BDC3C7'}
    bar_colors = [colors_map.get(l, '#7F8C8D') for l in subset['LABEL']]
    ax.barh(subset['LABEL'], subset['PCT'], color=bar_colors, edgecolor='black', linewidth=0.3)
    for i, (pct, count) in enumerate(zip(subset['PCT'], subset['TRIP_COUNT'])):
        if pct > 2:
            ax.text(pct + 0.5, i, f'{pct:.1f}% ({count/1e6:.1f}M)', va='center', fontsize=8)
    ax.set_xlabel('% of trips')
    ax.set_title(f'{title} - payment type distribution')
    ax.set_xlim(0, 75)
    ax.grid(True, alpha=0.2, axis='x')

plt.tight_layout()
plt.show()

---
## 8. Dataset similarity analysis

We measure similarity between dataset pairs using:
- **Cosine similarity** on hourly pickup distributions (temporal shape)
- **Cosine similarity** on pickup location distributions (spatial overlap)
- **Cosine similarity** on day-of-week distributions
- **Pearson correlation** on monthly trip volumes

In [ ]:
similarity = pd.concat([
    load_table('T3_HOURLY_SIMILARITY').assign(METRIC='hourly').rename(columns={'COSINE_SIMILARITY': 'SIMILARITY'}),
    load_table('T3_SPATIAL_SIMILARITY').assign(METRIC='spatial').rename(columns={'COSINE_SIMILARITY': 'SIMILARITY'}),
    load_table('T3_DOW_SIMILARITY').assign(METRIC='day_of_week').rename(columns={'COSINE_SIMILARITY': 'SIMILARITY'}),
    load_table('T3_MONTHLY_CORRELATION').assign(METRIC='monthly_corr').rename(columns={'PEARSON_CORRELATION': 'SIMILARITY'})
], ignore_index=True)

datasets_order = ['yellow', 'green', 'fhv', 'fhvhv']
metric_titles = {
    'hourly': 'Hourly pattern\n(cosine similarity)',
    'spatial': 'Spatial distribution\n(cosine similarity)',
    'day_of_week': 'Day-of-week pattern\n(cosine similarity)',
    'monthly_corr': 'Monthly volume\n(Pearson correlation)'
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, metric in zip(axes, ['hourly', 'spatial', 'day_of_week', 'monthly_corr']):
    matrix = np.eye(4)
    subset = similarity[similarity['METRIC'] == metric]
    for _, row in subset.iterrows():
        i = datasets_order.index(row['DATASET_A'])
        j = datasets_order.index(row['DATASET_B'])
        matrix[i, j] = row['SIMILARITY']
        matrix[j, i] = row['SIMILARITY']

    im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(['Yellow', 'Green', 'FHV', 'FHVHV'], fontsize=7, rotation=45)
    ax.set_yticklabels(['Yellow', 'Green', 'FHV', 'FHVHV'], fontsize=7)
    ax.set_title(metric_titles[metric], fontsize=9)

    for i in range(4):
        for j in range(4):
            color = 'white' if matrix[i,j] < 0.5 else 'black'
            ax.text(j, i, f'{matrix[i,j]:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.suptitle('Dataset similarity matrix - comparison across all metrics', fontsize=12, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Key similarity findings ===")
print("\nHourly temporal patterns:")
hourly_sim = similarity[similarity['METRIC'] == 'hourly'].sort_values('SIMILARITY', ascending=False)
for _, row in hourly_sim.iterrows():
    print(f"  {DATASET_LABELS[row['DATASET_A']]} vs {DATASET_LABELS[row['DATASET_B']]}: {row['SIMILARITY']:.4f}")

print("\nSpatial distribution (pickup locations):")
spatial_sim = similarity[similarity['METRIC'] == 'spatial'].sort_values('SIMILARITY', ascending=False)
for _, row in spatial_sim.iterrows():
    print(f"  {DATASET_LABELS[row['DATASET_A']]} vs {DATASET_LABELS[row['DATASET_B']]}: {row['SIMILARITY']:.4f}")

print("\nMonthly volume correlation:")
monthly_sim = similarity[similarity['METRIC'] == 'monthly_corr'].sort_values('SIMILARITY', ascending=False)
for _, row in monthly_sim.iterrows():
    print(f"  {DATASET_LABELS[row['DATASET_A']]} vs {DATASET_LABELS[row['DATASET_B']]}: {row['SIMILARITY']:.4f}")

print("\n=== Summary ===")
print("- All datasets share very similar HOURLY patterns (cosine > 0.98)")
print("- All datasets share very similar DAY-OF-WEEK patterns (cosine > 0.99)")
print("- SPATIAL overlap varies greatly: FHV/FHVHV are similar (0.85),")
print("  but Yellow/Green have very low overlap (0.08) - different service areas")
print("- Monthly volume: Green/Yellow most correlated (0.95) - both declining together")

---
## 9. Spatial overlap visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

yellow_top = set(top_pickup[top_pickup['DATASET'] == 'yellow'].nlargest(10, 'TRIP_COUNT')['PU_LOCATION_ID'].values)
green_top = set(top_pickup[top_pickup['DATASET'] == 'green'].nlargest(10, 'TRIP_COUNT')['PU_LOCATION_ID'].values)
fhv_top = set(top_pickup[top_pickup['DATASET'] == 'fhv'].nlargest(10, 'TRIP_COUNT')['PU_LOCATION_ID'].values)
fhvhv_top = set(top_pickup[top_pickup['DATASET'] == 'fhvhv'].nlargest(10, 'TRIP_COUNT')['PU_LOCATION_ID'].values)

all_top10 = sorted(yellow_top | green_top | fhv_top | fhvhv_top)
data_matrix = []
for loc in all_top10:
    row = []
    row.append(1 if loc in yellow_top else 0)
    row.append(1 if loc in green_top else 0)
    row.append(1 if loc in fhv_top else 0)
    row.append(1 if loc in fhvhv_top else 0)
    data_matrix.append(row)

data_matrix = np.array(data_matrix)
im = ax.imshow(data_matrix.T, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
ax.set_yticks(range(4))
ax.set_yticklabels(['Yellow', 'Green', 'FHV', 'FHVHV'])
ax.set_xticks(range(len(all_top10)))
ax.set_xticklabels([f'{int(l)}' for l in all_top10], rotation=45, fontsize=7)
ax.set_xlabel('Location zone ID')
ax.set_title('Top-10 pickup location overlap across datasets\n(dark = in top 10 for that dataset)')
plt.tight_layout()
plt.show()

print(f"\nZones in top-10 for ALL datasets: {yellow_top & green_top & fhv_top & fhvhv_top}")
print(f"Yellow-only zones (not in others' top-10): {yellow_top - green_top - fhv_top - fhvhv_top}")
print(f"Green-only zones (not in others' top-10): {green_top - yellow_top - fhv_top - fhvhv_top}")

---
## 10. Year-over-year growth rates

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for dataset in ['yellow', 'green', 'fhv', 'fhvhv']:
    subset = monthly_trend[monthly_trend['DATASET'] == dataset].copy()
    yearly = subset.groupby('PICKUP_YEAR')['TRIP_COUNT'].sum().reset_index()
    yearly = yearly[yearly['PICKUP_YEAR'] <= 2025]
    yearly['YOY_GROWTH'] = yearly['TRIP_COUNT'].pct_change() * 100
    yearly = yearly.dropna()
    ax.plot(yearly['PICKUP_YEAR'], yearly['YOY_GROWTH'],
            color=DATASET_COLORS[dataset], label=DATASET_LABELS[dataset],
            linewidth=2, marker='o', markersize=4)

ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Year')
ax.set_ylabel('Year-over-year growth (%)')
ax.set_title('Annual trip volume growth rate by dataset')
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. FHV statistics

In [ ]:
fhv_stats = load_table('T3_FHV_TRIP_STATS').sort_values('PICKUP_YEAR')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.bar(fhv_stats['PICKUP_YEAR'], fhv_stats['TRIP_COUNT'] / 1e6,
        color=DATASET_COLORS['fhv'], edgecolor='black', linewidth=0.3)
ax1.set_xlabel('Year')
ax1.set_ylabel('Trips (millions)')
ax1.set_title('FHV trip volume by year')
ax1.grid(True, alpha=0.2, axis='y')

ax2.plot(fhv_stats['PICKUP_YEAR'], fhv_stats['PCT_WITH_PU_LOCATION'],
         color='#3498DB', linewidth=2, marker='o', label='Pickup location available')
ax2.plot(fhv_stats['PICKUP_YEAR'], fhv_stats['PCT_WITH_DO_LOCATION'],
         color='#E74C3C', linewidth=2, marker='s', label='Dropoff location available')
ax2.set_xlabel('Year')
ax2.set_ylabel('% of trips')
ax2.set_title('FHV data completeness: location availability')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

---
## 12. Summary and key findings

In [ ]:
pairs = similarity.groupby(['DATASET_A', 'DATASET_B'])['SIMILARITY'].mean().reset_index()
pairs.columns = ['DATASET_A', 'DATASET_B', 'AVG_SIMILARITY']
pairs = pairs.sort_values('AVG_SIMILARITY', ascending=True)
pairs['PAIR_LABEL'] = [f"{DATASET_LABELS[a]} vs\n{DATASET_LABELS[b]}" for a, b in zip(pairs['DATASET_A'], pairs['DATASET_B'])]

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(pairs['AVG_SIMILARITY'])
ax.barh(pairs['PAIR_LABEL'], pairs['AVG_SIMILARITY'], color=colors, edgecolor='black', linewidth=0.3)
for i, val in enumerate(pairs['AVG_SIMILARITY']):
    ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Average similarity (across all 4 metrics)')
ax.set_title('Overall dataset pair similarity ranking\n(average of hourly, spatial, DOW, and monthly correlation)')
ax.set_xlim(0, 1.1)
ax.axvline(0.5, color='gray', linestyle=':', alpha=0.5)
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

print("\nMost similar pair:", pairs.iloc[-1]['PAIR_LABEL'].replace('\n', ' '), f"({pairs.iloc[-1]['AVG_SIMILARITY']:.3f})")
print("Least similar pair:", pairs.iloc[0]['PAIR_LABEL'].replace('\n', ' '), f"({pairs.iloc[0]['AVG_SIMILARITY']:.3f})")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
pair_list = [('yellow', 'green'), ('yellow', 'fhv'), ('yellow', 'fhvhv'),
             ('green', 'fhv'), ('green', 'fhvhv'), ('fhv', 'fhvhv')]

for ax, (d1, d2) in zip(axes.flatten(), pair_list):
    s1 = monthly_trend[monthly_trend['DATASET'] == d1].sort_values('MONTH_DATE')
    s2 = monthly_trend[monthly_trend['DATASET'] == d2].sort_values('MONTH_DATE')
    t1 = s1['TRIP_COUNT'] / s1['TRIP_COUNT'].max()
    t2 = s2['TRIP_COUNT'] / s2['TRIP_COUNT'].max()
    ax.plot(s1['MONTH_DATE'], t1.values, color=DATASET_COLORS[d1],
            label=DATASET_LABELS[d1], linewidth=1.2)
    ax.plot(s2['MONTH_DATE'], t2.values, color=DATASET_COLORS[d2],
            label=DATASET_LABELS[d2], linewidth=1.2)
    corr_row = similarity[(similarity['METRIC'] == 'monthly_corr') &
                          (((similarity['DATASET_A'] == d1) & (similarity['DATASET_B'] == d2)) |
                           ((similarity['DATASET_A'] == d2) & (similarity['DATASET_B'] == d1)))]
    corr_val = corr_row['SIMILARITY'].values[0] if len(corr_row) > 0 else 0
    ax.set_title(f"{DATASET_LABELS[d1]} vs {DATASET_LABELS[d2]}\n(r={corr_val:.3f})", fontsize=9)
    ax.legend(fontsize=7)
    ax.set_ylabel('Normalized volume')
    ax.grid(True, alpha=0.2)
    ax.set_ylim(0, 1.1)

plt.suptitle('Monthly volume trends (normalized) - pairwise comparison', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()